# Text Classification — Zero-Shot with Groq

In [ ]:
import os
import time
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from groq import Groq

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

env_candidates = [
    Path(".env"),
    Path("shot") / ".env",
]
loaded = False
for env_path in env_candidates:
    if env_path.exists():
        if load_dotenv is not None:
            load_dotenv(env_path)
        else:
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip().strip('"').strip("'")
        print(f"Loaded .env from: {env_path.resolve()}")
        loaded = True
        break
if not loaded:
    print("No .env file found in expected paths (.env, shot/.env).")

# -- CONFIG -----------------------------------------------------------------
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
MODEL = "llama-3.3-70b-versatile"
SLEEP_SEC = 1.0
# ---------------------------------------------------------------------------

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it to .env in this folder (or shot/.env).")

client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready")

✅ Groq client ready


## 1. Load Data
Load the 3 submission CSVs (same structure: `ID;Text`). Put them all in the same folder.

In [ ]:
print("FILES:", FILES)
print("len(all_df):", len(all_df))
print("len(predictions):", len(predictions))
print("len(results_df):", len(results_df))
print("len(final_df):", len(final_df))
print(final_df["ID"].head(10).tolist())

In [3]:
# ── Adjust paths to your files ──────────────────────────────────────────────
FILES = {

    "subm3": "subm3.csv",
}

dfs = {}
for name, path in FILES.items():
    df = pd.read_csv(path, sep=";", encoding="utf-8-sig")
    dfs[name] = df
    print(f"{name}: {len(df)} rows — columns: {list(df.columns)}")

# Combine all into one DataFrame for batch processing
all_df = pd.concat(dfs.values(), ignore_index=True)
print(f"\nTotal rows: {len(all_df)}")
all_df.head()

subm3: 150 rows — columns: ['ID', 'Text']

Total rows: 150


,ID,Text
0,D2-126,The reality about the places that diamonds are...
1,D2-127,Geothermobarometric calculations for a worldwi...
2,D2-128,Diamonds are formed deep within the Earth’s ma...
3,D2-129,Diamond is a solid form of the element carbon ...
4,D2-130,Diamonds are formed deep within the Earth unde...


## 2. Define Your Labels

Paste the full list of valid labels here. These come from your training set (the `~300 labels` you mentioned).  
The format should be a Python list of strings.

In [ ]:
# -- PASTE YOUR LABELS HERE --------------------------------------------------
LABELS = [
    "Human",
    "OpenAI",
    "Google",
    "Meta",
    "Anthropic",
]

LABELS = sorted({str(l).strip() for l in LABELS if str(l).strip()})
VALID_LABELS_LOWER = {l.lower(): l for l in LABELS}

print(f"Total labels defined: {len(LABELS)}")
print("First 5:", LABELS[:5])

Total labels defined: 5
First 5: ['Human', 'OpenAI', 'Google', 'Meta', 'Anthropic']


## 3. Zero-Shot Classification Function

The prompt gives the model:
- The text to classify
- The full list of valid labels
- Strict instruction to output only the label name

In [ ]:
LABELS_STR = "\n".join(f"- {l}" for l in LABELS)

SYSTEM_PROMPT = """You are a precise text classification assistant.
Classify each text into exactly one label from the provided label list.
Use only labels from the list and output only the label name (no extra text)."""


def normalize_pred_label(pred: str) -> str:
    pred_clean = str(pred).strip().strip('"\'`').rstrip(".,;:").strip()
    return VALID_LABELS_LOWER.get(pred_clean.lower(), pred_clean)


def classify_zero_shot(text: str) -> str:
    """Classify a single text using zero-shot prompting."""
    text = "" if text is None else str(text)
    user_prompt = f"""Valid labels:

{LABELS_STR}

Text to classify:
\"\"\"
{text[:3000]}
\"\"\"

Return exactly one label from the valid labels list."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=20,
    )
    raw = response.choices[0].message.content
    return normalize_pred_label(raw)


# Quick test
sample_text = all_df["Text"].iloc[0]
print("Sample text (first 200 chars):", sample_text[:200])
print("\nPredicted label:", classify_zero_shot(sample_text))

Sample text (first 200 chars): The reality about the places that diamonds are found is that our planet might host enormous quantities of diamonds below its rocky surface. However, it is only the diamonds that are found at the surfa

Predicted label: Human


## 4. Run Inference on All Rows

In [ ]:
predictions = []
errors      = []

for idx, row in tqdm(all_df.iterrows(), total=len(all_df), desc="Classifying"):
    try:
        label = classify_zero_shot(row["Text"])
        predictions.append({"ID": row["ID"], "Label": label})
    except Exception as e:
        print(f"Error on ID {row['ID']}: {e}")
        errors.append(row["ID"])
        predictions.append({"ID": row["ID"], "Label": "ERROR"})
    time.sleep(SLEEP_SEC)

print(f"\nDone. Errors: {len(errors)}")
if errors:
    print("Failed IDs:", errors)

Classifying: 100%|██████████| 150/150 [07:18<00:00,  2.92s/it]


✅ Done. Errors: 0


## 5. Post-process: Snap to Closest Valid Label
If the model returns something not exactly in the label list, we find the closest match.

In [ ]:
from difflib import get_close_matches

results_df = pd.DataFrame(predictions)

def snap_label(pred: str, valid_labels: list, cutoff: float = 0.6) -> str:
    """Return the closest valid label, or the original if no good match."""
    pred = normalize_pred_label(pred)
    if pred in valid_labels:
        return pred
    matches = get_close_matches(pred, valid_labels, n=1, cutoff=cutoff)
    return matches[0] if matches else pred

results_df["Label_snapped"] = results_df["Label"].apply(
    lambda x: snap_label(x, LABELS)
)

# Show any corrections
corrections = results_df[results_df["Label"] != results_df["Label_snapped"]]
print(f"Corrections made: {len(corrections)}")
corrections[["ID", "Label", "Label_snapped"]].head(10)

Corrections made: 0


,ID,Label,Label_snapped


## 6. Save Submissions
One output file per dataset.

In [15]:
len(final_df)

150

In [ ]:
final_df = results_df[["ID", "Label_snapped"]].rename(columns={"Label_snapped": "Label"})

out_path = "submission_zeroshot_all.csv"
final_df.to_csv(out_path, sep=";", index=False)
print(f"Saved {len(final_df)} rows -> {out_path}")

final_df.head(10)

Saved 150 rows -> submission_zeroshot_subm3.csv


,ID,Label
0,D2-126,Human
1,D2-127,Human
2,D2-128,Human
3,D2-129,Human
4,D2-130,Human
5,D2-131,Human
6,D2-132,Human
7,D2-133,Human
8,D2-134,Human
9,D2-135,Human


## 7. Quick Analysis

In [ ]:
print("Label distribution (top 20):")
print(final_df["Label"].value_counts().head(20))

# Check for any unexpected labels
unknown = set(final_df["Label"].unique()) - set(LABELS)
if unknown:
    print(f"\nLabels NOT in your list: {unknown}")
else:
    print("\nAll predicted labels are valid!")

Label distribution (top 20):
Label
Human     101
OpenAI     48
Google      1
Name: count, dtype: int64

✅ All predicted labels are valid!



## Evaluation on `data-exemplo.csv`



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from difflib import get_close_matches

# -- Load labeled evaluation set ---------------------------------------------
EVAL_CANDIDATES = ["dataset-exemplos.csv", "data-exemplo.csv"]
eval_path = next((p for p in EVAL_CANDIDATES if os.path.exists(p)), None)
if eval_path is None:
    raise FileNotFoundError(f"Could not find any eval file in: {EVAL_CANDIDATES}")

eval_df = pd.read_csv(eval_path, sep=";", encoding="utf-8-sig")
print(f"Evaluation file: {eval_path}")
print(f"Evaluation set: {len(eval_df)} rows")
print(f"Columns: {list(eval_df.columns)}")

# Auto-detect label column name (handles 'Label', 'label', 'Topic', etc.)
label_col = next(
    (c for c in eval_df.columns if c.lower() in ["label", "topic", "class", "category"]),
    eval_df.columns[-1],
)
print(f"Using '{label_col}' as the label column")
print(f"Unique true labels: {eval_df[label_col].nunique()}")
eval_df.head()

Evaluation set: 125 rows
Columns: ['ID', 'Text', 'Label']
Using 'Label' as the label column
Unique true labels: 5


,ID,Text,Label
0,D1-1,"It is an approximation useful in chemistry, bu...",Human
1,D1-2,"PET scanning, or Positron Emission Tomography,...",Meta
2,D1-3,Positron Emission Tomography (PET) scanning is...,Google
3,D1-4,Thermonuclear fusion is the process of combini...,Meta
4,D1-5,"These nutrients are needed to keep bones, teet...",Human


In [ ]:
# ── Run zero-shot predictions on eval set ───────────────────────────────────
eval_predictions = []
eval_errors      = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating (zero-shot)"):
    try:
        label = classify_zero_shot(row["Text"])
        eval_predictions.append({"ID": row["ID"], "Pred": label, "True": row[label_col]})
    except Exception as e:
        print(f"Error on ID {row['ID']}: {e}")
        eval_errors.append(row["ID"])
        eval_predictions.append({"ID": row["ID"], "Pred": "ERROR", "True": row[label_col]})
    time.sleep(SLEEP_SEC)

print(f"\nEvaluation done. Errors: {len(eval_errors)}")
eval_results = pd.DataFrame(eval_predictions)

Evaluating (zero-shot): 100%|██████████| 125/125 [06:23<00:00,  3.07s/it]


✅ Evaluation done. Errors: 0


In [ ]:
# -- Snap predictions to valid labels ----------------------------------------
eval_results["Pred_snapped"] = eval_results["Pred"].apply(
    lambda x: snap_label(x, LABELS)
)

y_true = eval_results["True"].tolist()
y_pred = eval_results["Pred_snapped"].tolist()

acc = accuracy_score(y_true, y_pred)
print(f"\n{'='*50}")
print(f"  Zero-Shot Accuracy: {acc:.2%}  ({int(acc*len(y_true))}/{len(y_true)} correct)")
print(f"{'='*50}\n")

# Per-label breakdown
print(classification_report(y_true, y_pred, zero_division=0))


  Zero-Shot Accuracy: 39.20%  (49/125 correct)

              precision    recall  f1-score   support

   Anthropic       0.00      0.00      0.00        23
      Google       0.00      0.00      0.00        16
       Human       0.43      0.92      0.59        52
        Meta       0.00      0.00      0.00        17
      OpenAI       0.07      0.06      0.06        17

    accuracy                           0.39       125
   macro avg       0.10      0.20      0.13       125
weighted avg       0.19      0.39      0.25       125



In [ ]:
# ── Confusion matrix (best for ≤ 30 labels; readable heatmap otherwise) ──────
present_labels = sorted(set(y_true + y_pred))
cm = confusion_matrix(y_true, y_pred, labels=present_labels)

fig_size = max(12, len(present_labels) * 0.6)
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
sns.heatmap(
    cm,
    annot=True, fmt="d", cmap="Blues",
    xticklabels=present_labels,
    yticklabels=present_labels,
    ax=ax,
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title("Confusion Matrix — Zero-Shot", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig("confusion_matrix_zeroshot.png", dpi=150)
plt.show()
print("Saved → confusion_matrix_zeroshot.png")

In [ ]:
# ── Show worst-performing labels ─────────────────────────────────────────────
report_dict = {}
from sklearn.metrics import classification_report
report_raw = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

per_label = (
    pd.DataFrame(report_raw)
    .T
    .drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")
    .sort_values("f1-score")
)

print("\nWorst-performing labels (lowest F1):")
print(per_label.head(10).to_string())

print("\nBest-performing labels (highest F1):")
print(per_label.tail(10).to_string())

In [ ]:
# ── Show wrong predictions for manual inspection ──────────────────────────────
wrong = eval_results[eval_results["True"] != eval_results["Pred_snapped"]].copy()
wrong = wrong.merge(eval_df[["ID", "Text"]], on="ID", how="left")
wrong["Text_preview"] = wrong["Text"].str[:120]

print(f"Wrong predictions: {len(wrong)} / {len(eval_results)}")
wrong[["ID", "True", "Pred_snapped", "Text_preview"]].head(20)

In [ ]:
# Save for comparison in the few-shot notebook
eval_results.to_csv("eval_zeroshot_results.csv", sep=";", index=False)
print("Saved eval results -> eval_zeroshot_results.csv")
print(f"\nZero-Shot summary: Accuracy = {acc:.2%}")